## Perth House Price Prediction

Given *data about houses in Perth*, let's try to predict the **price** of a given house.

We will use three different linear regression models to make our predictions.

Data source: https://www.kaggle.com/datasets/syuzai/perth-house-prices

### Importing Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso

import warnings
warnings.simplefilter(action='ignore')

In [2]:
data = pd.read_csv('archive/all_perth_310121.csv')
data

,ADDRESS,SUBURB,PRICE,BEDROOMS,BATHROOMS,GARAGE,LAND_AREA,FLOOR_AREA,BUILD_YEAR,CBD_DIST,NEAREST_STN,NEAREST_STN_DIST,DATE_SOLD,POSTCODE,LATITUDE,LONGITUDE,NEAREST_SCH,NEAREST_SCH_DIST,NEAREST_SCH_RANK
0,1 Acorn Place,South Lake,565000,4,2,2.0,600,160,2003.0,18300,Cockburn Central Station,1800,09-2018\r,6164,-32.115900,115.842450,LAKELAND SENIOR HIGH SCHOOL,0.828339,NaN
1,1 Addis Way,Wandi,365000,3,2,2.0,351,139,2013.0,26900,Kwinana Station,4900,02-2019\r,6167,-32.193470,115.859554,ATWELL COLLEGE,5.524324,129.0
2,1 Ainsley Court,Camillo,287000,3,1,1.0,719,86,1979.0,22600,Challis Station,1900,06-2015\r,6111,-32.120578,115.993579,KELMSCOTT SENIOR HIGH SCHOOL,1.649178,113.0
3,1 Albert Street,Bellevue,255000,2,1,2.0,651,59,1953.0,17900,Midland Station,3600,07-2018\r,6056,-31.900547,116.038009,SWAN VIEW SENIOR HIGH SCHOOL,1.571401,NaN
4,1 Aman Place,Lockridge,325000,4,1,2.0,466,131,1998.0,11200,Bassendean Station,2000,11-2016\r,6054,-31.885790,115.947780,KIARA COLLEGE,1.514922,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33651,9C Gold Street,South Fremantle,1040000,4,3,2.0,292,245,2013.0,16100,Fremantle Station,1500,03-2016\r,6162,-32.064580,115.751820,CHRISTIAN BROTHERS' COLLEGE,1.430350,49.0
33652,9C Pycombe Way,Westminster,410000,3,2,2.0,228,114,NaN,9600,Stirling Station,4600,02-2017\r,6061,-31.867055,115.841403,JOHN SEPTIMUS ROE ANGLICAN COMMUNITY SCHOOL,1.679644,35.0
33653,9D Pycombe Way,Westminster,427000,3,2,2.0,261,112,NaN,9600,Stirling Station,4600,02-2017\r,6061,-31.866890,115.841418,JOHN SEPTIMUS ROE ANGLICAN COMMUNITY SCHOOL,1.669159,35.0
33654,9D Shalford Way,Girrawheen,295000,3,1,2.0,457,85,1974.0,12600,Warwick Station,4400,10-2016\r,6064,-31.839680,115.842410,GIRRAWHEEN SENIOR HIGH SCHOOL,0.358494,NaN


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33656 entries, 0 to 33655
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ADDRESS           33656 non-null  object 
 1   SUBURB            33656 non-null  object 
 2   PRICE             33656 non-null  int64  
 3   BEDROOMS          33656 non-null  int64  
 4   BATHROOMS         33656 non-null  int64  
 5   GARAGE            31178 non-null  float64
 6   LAND_AREA         33656 non-null  int64  
 7   FLOOR_AREA        33656 non-null  int64  
 8   BUILD_YEAR        30501 non-null  float64
 9   CBD_DIST          33656 non-null  int64  
 10  NEAREST_STN       33656 non-null  object 
 11  NEAREST_STN_DIST  33656 non-null  int64  
 12  DATE_SOLD         33656 non-null  object 
 13  POSTCODE          33656 non-null  int64  
 14  LATITUDE          33656 non-null  float64
 15  LONGITUDE         33656 non-null  float64
 16  NEAREST_SCH       33656 non-null  object

### Preprocessing

In [65]:
def preprocess_inputs(df):
    df = df.copy()

    # Drop high cardinality address column
    df = df.drop('ADDRESS', axis=1)

    # Drop columns with more than 25% missing values
    df = df.drop('NEAREST_SCH_RANK', axis=1)

    # Convert GARAGE into object dtype
    df['GARAGE'] = df['GARAGE'].astype('str')

    # Convert POSTCODE into object dtype
    df['POSTCODE'] = df['POSTCODE'].astype('str')    

    df['BUILD_YEAR'] = df['BUILD_YEAR'].fillna(df['BUILD_YEAR'].median())

    # Extract date features
    df['DATE_SOLD'] = pd.to_datetime(df['DATE_SOLD'])
    df['DATE_YEAR'] = df['DATE_SOLD'].dt.year
    df['DATE_MONTH'] = df['DATE_SOLD'].dt.month
    df = df.drop('DATE_SOLD', axis=1)
    
    # One-hot encode nominal features
    for column in df.select_dtypes('object').columns:
        dummies = pd.get_dummies(df[column], prefix=column, dtype=int)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop(column, axis=1)

    # Split df into X and y
    y = df['PRICE'].copy()
    X = df.drop('PRICE', axis=1).copy()

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=1)

    # Scale X
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

    return X_train, X_test, y_train, y_test

In [66]:
X_train, X_test, y_train, y_test = preprocess_inputs(data)

In [67]:
X_train

,BEDROOMS,BATHROOMS,LAND_AREA,FLOOR_AREA,BUILD_YEAR,CBD_DIST,NEAREST_STN_DIST,LATITUDE,LONGITUDE,NEAREST_SCH_DIST,...,NEAREST_SCH_TRINITY COLLEGE,NEAREST_SCH_URSULA FRAYNE CATHOLIC COLLEGE,NEAREST_SCH_WANNEROO SECONDARY COLLEGE,NEAREST_SCH_WARNBRO COMMUNITY HIGH SCHOOL,NEAREST_SCH_WARWICK SENIOR HIGH SCHOOL,NEAREST_SCH_WESLEY COLLEGE,NEAREST_SCH_WILLETTON SENIOR HIGH SCHOOL,NEAREST_SCH_WOODVALE SECONDARY COLLEGE,NEAREST_SCH_YANCHEP SECONDARY COLLEGE,NEAREST_SCH_YOUTH FUTURES COMMUNITY SCHOOL
25967,-0.871049,-1.406198,-0.131130,-1.350650,-0.109780,0.122923,-0.892045,1.058986,-0.809147,-0.378097,...,-0.026872,-0.04471,-0.128893,-0.077872,-0.095518,-0.050951,-0.046578,-0.058737,-0.097534,-0.123158
28153,-0.871049,0.310844,-0.155586,-0.598766,0.894532,-1.509302,-0.808748,0.164319,-0.303484,-0.741544,...,-0.026872,-0.04471,-0.128893,-0.077872,-0.095518,-0.050951,-0.046578,-0.058737,-0.097534,-0.123158
33655,-0.871049,-1.406198,-0.155972,-1.225336,0.241729,-0.274105,-0.631991,0.443085,1.134626,-0.434196,...,-0.026872,-0.04471,-0.128893,-0.077872,-0.095518,-0.050951,-0.046578,-0.058737,-0.097534,-0.123158
211,1.786428,0.310844,-0.137952,-0.000043,1.095394,-0.335865,-0.587802,0.737233,-0.942817,-0.523915,...,-0.026872,-0.04471,-0.128893,-0.077872,-0.095518,-0.050951,-0.046578,-0.058737,-0.097534,-0.123158
23604,0.457690,0.310844,-0.125402,-0.292442,-0.662151,-0.397625,-0.190098,0.648038,-1.037757,0.082063,...,-0.026872,-0.04471,-0.128893,-0.077872,-0.095518,-0.050951,-0.046578,-0.058737,-0.097534,-0.123158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7813,-0.871049,-1.406198,-0.142393,-1.406346,-1.314954,-1.006400,-0.521518,-0.370176,-0.296301,-0.477396,...,-0.026872,-0.04471,-0.128893,-0.077872,-0.095518,-0.050951,-0.046578,-0.058737,-0.097534,-0.123158
32511,0.457690,0.310844,-0.131001,-0.473452,-0.159995,1.834553,-0.388950,-1.920371,-1.222212,-0.707221,...,-0.026872,-0.04471,-0.128893,-0.077872,-0.095518,-0.050951,-0.046578,-0.058737,-0.097534,-0.123158
5192,0.457690,0.310844,-0.141556,-0.751927,0.241729,0.696407,-0.631991,-1.166918,1.061257,-0.428093,...,-0.026872,-0.04471,-0.128893,-0.077872,-0.095518,-0.050951,-0.046578,-0.058737,-0.097534,-0.123158
12172,-0.871049,0.310844,-0.151595,-0.682308,0.543023,-0.865235,-0.764559,-0.451090,-0.296152,-0.130051,...,-0.026872,-0.04471,-0.128893,-0.077872,-0.095518,-0.050951,-0.046578,-0.058737,-0.097534,-0.123158


In [68]:
y_train

25967     486000
28153     855000
33655     295000
211       730000
23604     825000
          ...   
7813     2000000
32511     322500
5192      355000
12172     725000
33003     428500
Name: PRICE, Length: 23559, dtype: int64

### Training

In [69]:
models = {
    '                Linear Regression': LinearRegression(),
    'Ridge (L2-Regularized) Regression': Ridge(),
    'Lasso (L1-Regularized) Regression': Lasso()
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name + " trained.")

                Linear Regression trained.
Ridge (L2-Regularized) Regression trained.
Lasso (L1-Regularized) Regression trained.


### Results

In [70]:
for name, model in models.items():
    print(name + " R^2 Score: {:.5f}".format(model.score(X_test, y_test)))

                Linear Regression R^2 Score: 0.77743
Ridge (L2-Regularized) Regression R^2 Score: 0.77744
Lasso (L1-Regularized) Regression R^2 Score: 0.77741


In [71]:
ridge_model = Ridge(alpha=10.0)
ridge_model.fit(X_train, y_train)
print("Ridge" + " R^2 Score: {:.5f}".format(ridge_model.score(X_test, y_test)))

Ridge R^2 Score: 0.77745
